# 🚚 Delhivery Logistics Network — Data Exploration
### Optimizing Delivery ETAs with Graph-Based Network Intelligence

**Objective:** Understand the raw structure of Delhivery's trip data before any modeling.  
We begin by asking the most basic but most important question:  
*What does this data actually look like, and does it confirm the problem we're trying to solve?*

---

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("✅ All libraries loaded successfully")
print(f"   pandas     : {pd.__version__}")
print(f"   numpy      : {np.__version__}")
print(f"   matplotlib : {plt.matplotlib.__version__}")

In [ ]:
import os

# Create output folders if they don't exist
os.makedirs('../outputs/visualisations', exist_ok=True)
os.makedirs('../outputs/model_results', exist_ok=True)

print("✅ Output folders ready")

In [ ]:
# Load dataset
df = pd.read_csv('../data/delivery_data.csv')

print("=" * 55)
print("         DATASET LOADED SUCCESSFULLY")
print("=" * 55)
print(f"\n  Total Rows    : {df.shape[0]:,}")
print(f"  Total Columns : {df.shape[1]}")
print(f"\n  Memory Usage  : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\n" + "=" * 55)

## 🔍 Step 2 — First Look at the Data

Let's see what the first few rows actually look like.  
This is where we get familiar with the structure before anything else.

In [ ]:
print("First 5 rows of the dataset:\n")
df.head()

In [ ]:
print("Last 5 rows of the dataset:\n")
df.tail()

## 🗂️ Step 3 — Column Names and Data Types

Understanding what each column is and what type of data it holds.  
We're specifically looking for:
- Columns stored as wrong types (e.g. timestamps as strings)
- Columns that seem redundant or unclear
- Which columns will become our graph nodes and edges

In [ ]:
print("Column Names and Data Types:\n")
print("-" * 45)
for col, dtype in zip(df.columns, df.dtypes):
    print(f"  {col:<40} {str(dtype)}")
print("-" * 45)
print(f"\nTotal columns: {len(df.columns)}")

## 🔎 Step 4 — Missing Values Analysis

Missing data in logistics datasets is common and meaningful.  
A missing OSRM time might mean the route was unplanned.  
A missing actual time might mean the trip is still in transit.  
We need to understand both the count and the percentage.

In [ ]:
# Compute missing values
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count' : missing_count,
    'Missing %'     : missing_pct
}).sort_values('Missing %', ascending=False)

# Only show columns that have at least some missing values
missing_df = missing_df[missing_df['Missing Count'] > 0]

if len(missing_df) == 0:
    print("✅ No missing values found in the dataset!")
else:
    print(f"⚠️  {len(missing_df)} columns have missing values:\n")
    print(missing_df.to_string())

In [ ]:
# Visualize missing values
if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    bars = ax.barh(
        missing_df.index,
        missing_df['Missing %'],
        color='#E05C5C',
        edgecolor='white',
        height=0.6
    )
    
    # Add value labels on bars
    for bar, val in zip(bars, missing_df['Missing %']):
        ax.text(
            bar.get_width() + 0.3,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%',
            va='center',
            fontsize=10,
            color='#333333'
        )
    
    ax.set_xlabel('Missing Percentage (%)', fontsize=12)
    ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlim(0, max(missing_df['Missing %']) * 1.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../outputs/visualisations/missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Chart saved to outputs/visualisations/")
else:
    print("No missing values to visualize.")

## 📊 Step 5 — Basic Statistical Summary

We look at distributions of all numeric columns.  
Pay special attention to:
- Min/max values that seem unrealistic
- Large gaps between mean and median (skewed data)
- Any negative values in time or distance columns

In [ ]:
print("Statistical Summary of Numeric Columns:\n")
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

## 🏭 Step 6 — Understanding the Logistics Network

Before building the graph, we need to know the scale of the network.  
How many unique hubs exist? How many corridors?  
This tells us how complex the graph will be.

In [ ]:
# Identify source and destination columns
# Adjust column names below if they differ in your dataset
source_col = 'source_center'
dest_col   = 'destination_center'
route_col  = 'route_type'

# Check if these columns exist
for col in [source_col, dest_col, route_col]:
    if col in df.columns:
        print(f"✅ Found column: '{col}'")
    else:
        print(f"❌ Column not found: '{col}' — check your column names above")

In [ ]:
# Network scale analysis
unique_sources = df[source_col].nunique()
unique_dests   = df[dest_col].nunique()
all_hubs       = pd.concat([df[source_col], df[dest_col]]).nunique()
unique_corridors = df.groupby([source_col, dest_col]).ngroups

print("=" * 50)
print("       LOGISTICS NETWORK — SCALE OVERVIEW")
print("=" * 50)
print(f"\n  Unique Source Hubs       : {unique_sources:,}")
print(f"  Unique Destination Hubs  : {unique_dests:,}")
print(f"  Total Unique Hubs (Nodes): {all_hubs:,}")
print(f"  Unique Corridors (Edges) : {unique_corridors:,}")
print(f"  Total Trip Records       : {len(df):,}")
print("\n" + "=" * 50)
print(f"\n  Average trips per corridor: {len(df)/unique_corridors:.1f}")

## 🚛 Step 7 — Route Type Distribution

Delhivery operates two route types: **FTL** (Full Truck Load) and **Carting**.  
Understanding their split is important because:
- They have fundamentally different delay profiles
- The FTL vs Carting decision framework in Phase 6 depends on this

In [ ]:
route_counts = df[route_col].value_counts()
route_pct    = df[route_col].value_counts(normalize=True) * 100

print("Route Type Distribution:\n")
print("-" * 35)
for route, count, pct in zip(route_counts.index, route_counts.values, route_pct.values):
    bar = "█" * int(pct / 2)
    print(f"  {route:<12} {count:>8,} trips  ({pct:.1f}%)  {bar}")
print("-" * 35)

In [ ]:
# Pie chart
fig, ax = plt.subplots(figsize=(7, 7))

colors = ['#4A90D9', '#E07B54']
wedges, texts, autotexts = ax.pie(
    route_counts.values,
    labels=route_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)

for text in texts:
    text.set_fontsize(13)
for autotext in autotexts:
    autotext.set_fontsize(12)
    autotext.set_fontweight('bold')
    autotext.set_color('white')

ax.set_title('Route Type Distribution\n(FTL vs Carting)', fontsize=15, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('../outputs/visualisations/route_type_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

## ⏱️ Step 8 — The Core Signal: Actual vs OSRM Time

This is the most important analysis in Step 1.  
The entire project exists because OSRM underestimates actual delivery time.  
Here we verify that claim directly from the data.

We compute the **delay ratio** = `actual_time / osrm_time`  
- A ratio of 1.0 means perfect prediction  
- A ratio > 1.0 means OSRM underestimated (actual took longer)  
- A ratio > 1.2 means chronic delay (20%+ over estimate)

In [ ]:
# Correct column names from dataset
actual_time_col = 'actual_time'
osrm_time_col   = 'osrm_time'

for col in [actual_time_col, osrm_time_col]:
    if col in df.columns:
        print(f"✅ Found: '{col}'")
    else:
        print(f"❌ Not found: '{col}'")

In [ ]:
# Compute delay ratio — the core metric of this entire project
df['delay_ratio'] = df[actual_time_col] / df[osrm_time_col]

# Remove infinite or null ratios (division by zero edge cases)
df['delay_ratio'] = df['delay_ratio'].replace([np.inf, -np.inf], np.nan)

print("Delay Ratio (Actual / OSRM) — Summary Statistics:\n")
print("-" * 45)
stats = df['delay_ratio'].describe()
for stat, val in stats.items():
    print(f"  {stat:<10} : {val:.4f}")
print("-" * 45)

mean_ratio = df['delay_ratio'].mean()
if mean_ratio > 1.0:
    print(f"\n✅ CONFIRMED: Mean delay ratio is {mean_ratio:.3f}")
    print(f"   On average, actual delivery takes {(mean_ratio-1)*100:.1f}% longer than OSRM predicts.")
    print(f"   This validates the core problem statement.")
else:
    print(f"\n⚠️  Mean ratio is {mean_ratio:.3f} — review your column mappings.")

In [ ]:
# Chronic delay analysis
chronic_threshold = 1.2
chronic = df[df['delay_ratio'] > chronic_threshold]

total       = len(df.dropna(subset=['delay_ratio']))
chronic_cnt = len(chronic)
chronic_pct = chronic_cnt / total * 100

print("=" * 50)
print("        CHRONIC DELAY ANALYSIS")
print("=" * 50)
print(f"\n  Total trips analyzed     : {total:,}")
print(f"  Chronic delay trips      : {chronic_cnt:,}")
print(f"  Chronic delay rate       : {chronic_pct:.1f}%")
print(f"\n  Definition: Actual time > OSRM by more than 20%")
print("\n" + "=" * 50)

In [ ]:
# Distribution plot of delay ratio
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — histogram
axes[0].hist(
    df['delay_ratio'].dropna(),
    bins=60,
    color='#4A90D9',
    edgecolor='white',
    alpha=0.85
)
axes[0].axvline(1.0, color='green',  linewidth=2, linestyle='--', label='Perfect prediction (1.0)')
axes[0].axvline(1.2, color='red',    linewidth=2, linestyle='--', label='Chronic delay threshold (1.2)')
axes[0].axvline(df['delay_ratio'].mean(), color='orange', linewidth=2, linestyle='-', label=f"Mean ({df['delay_ratio'].mean():.2f})")
axes[0].set_xlabel('Delay Ratio (Actual / OSRM)', fontsize=12)
axes[0].set_ylabel('Number of Trips', fontsize=12)
axes[0].set_title('Distribution of Delay Ratio', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Right — delay ratio by route type
route_delay = df.groupby(route_col)['delay_ratio'].mean().sort_values(ascending=False)
colors_bar  = ['#E07B54' if r == route_delay.index[0] else '#4A90D9' for r in route_delay.index]

bars = axes[1].bar(
    route_delay.index,
    route_delay.values,
    color=colors_bar,
    edgecolor='white',
    width=0.5
)
axes[1].axhline(1.0, color='green', linewidth=2, linestyle='--', label='Perfect prediction')
for bar, val in zip(bars, route_delay.values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{val:.3f}',
        ha='center',
        fontsize=11,
        fontweight='bold'
    )
axes[1].set_xlabel('Route Type', fontsize=12)
axes[1].set_ylabel('Mean Delay Ratio', fontsize=12)
axes[1].set_title('Mean Delay Ratio by Route Type', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('OSRM vs Actual Delivery Time Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/visualisations/delay_ratio_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

## 📋 Step 9 — Key Findings Summary

Before moving to data cleaning, we document everything we've learned.  
These findings will directly shape every decision in Steps 2 through 7.

In [ ]:
print("=" * 60)
print("         DATA EXPLORATION — KEY FINDINGS")
print("=" * 60)

print(f"""
DATASET OVERVIEW
  • Total records       : {len(df):,}
  • Total columns       : {df.shape[1]}
  • Memory footprint    : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB

NETWORK STRUCTURE
  • Unique hubs (nodes) : {all_hubs:,}
  • Unique corridors    : {unique_corridors:,}
  • Avg trips/corridor  : {len(df)/unique_corridors:.1f}

ROUTE TYPE SPLIT
  • FTL trips           : {route_counts.get('FTL', 0):,} ({route_counts.get('FTL', 0)/len(df)*100:.1f}%)
  • Carting trips       : {route_counts.get('Carting', 0):,} ({route_counts.get('Carting', 0)/len(df)*100:.1f}%)

DELAY ANALYSIS
  • Mean delay ratio    : {df['delay_ratio'].mean():.3f}
  • Median delay ratio  : {df['delay_ratio'].median():.3f}
  • Chronic delay rate  : {chronic_pct:.1f}% of trips
  • OSRM underestimates : {'YES ✅' if df['delay_ratio'].mean() > 1 else 'NO ❌'}

MISSING VALUES
  • Columns with nulls  : {len(missing_df)}
""")

print("=" * 60)
print("  NEXT STEP → 02_data_cleaning.ipynb")
print("=" * 60)

## 📦 Step 10 — Understanding Trip Segments vs Full Journeys

Each row in this dataset is a **segment** — one hop of a multi-leg journey.  
Multiple rows share the same `trip_uuid` — together they form one complete trip.  
This is critical to understand before building the graph.

In [ ]:
# Understand segment vs journey structure
total_rows     = len(df)
unique_trips   = df['trip_uuid'].nunique()
unique_routes  = df['route_schedule_uuid'].nunique()
avg_segments   = total_rows / unique_trips

print("=" * 55)
print("     TRIP SEGMENT STRUCTURE ANALYSIS")
print("=" * 55)
print(f"\n  Total segment rows       : {total_rows:,}")
print(f"  Unique trips (trip_uuid) : {unique_trips:,}")
print(f"  Unique routes            : {unique_routes:,}")
print(f"  Avg segments per trip    : {avg_segments:.2f}")
print("\n" + "=" * 55)
print(f"""
  💡 Insight:
  Each trip has on average {avg_segments:.1f} segments.
  This means shipments travel through {avg_segments:.1f} hubs 
  on average before reaching their destination.
  These hubs become the NODES in our graph.
""")

## 📍 Step 11 — Geographic Coverage

Understanding which states and regions are covered.  
Delhivery operates pan-India — let's see the distribution  
of source hubs across the network.

In [ ]:
# Extract state from hub names
# Hub names follow pattern: IND{pincode}AAA or city_name (State)
# State appears in brackets in source_name column

df['source_state'] = df['source_name'].str.extract(r'\(([^)]+)\)')
df['dest_state']   = df['destination_name'].str.extract(r'\(([^)]+)\)')

source_state_counts = df['source_state'].value_counts().head(15)

print("Top 15 States by Number of Source Hub Trips:\n")
print("-" * 45)
for state, count in source_state_counts.items():
    bar = "█" * int(count / source_state_counts.max() * 30)
    print(f"  {state:<20} {count:>7,}  {bar}")
print("-" * 45)
print(f"\n  Total unique states: {df['source_state'].nunique()}")

In [ ]:
# Bar chart of top states
fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(
    source_state_counts.index[::-1],
    source_state_counts.values[::-1],
    color='#4A90D9',
    edgecolor='white',
    height=0.6
)

for bar, val in zip(bars, source_state_counts.values[::-1]):
    ax.text(
        bar.get_width() + source_state_counts.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'{val:,}',
        va='center',
        fontsize=9,
        color='#333333'
    )

ax.set_xlabel('Number of Trip Segments', fontsize=12)
ax.set_title('Top 15 States by Trip Volume', fontsize=14, fontweight='bold', pad=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/visualisations/state_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

## ⏰ Step 12 — Time Dimension Analysis

The dataset has timestamps for trip creation, OD start, and OD end.  
We extract time-of-day patterns to understand when delays are worst.  
This will later become a key feature in our graph edge weights.

In [ ]:
# Parse timestamps
df['trip_creation_time'] = pd.to_datetime(df['trip_creation_time'])
df['od_start_time']      = pd.to_datetime(df['od_start_time'])
df['od_end_time']        = pd.to_datetime(df['od_end_time'])

# Extract time features
df['hour_of_day']   = df['od_start_time'].dt.hour
df['day_of_week']   = df['od_start_time'].dt.day_name()
df['month']         = df['od_start_time'].dt.month

# Create time of day buckets
def time_bucket(hour):
    if 5 <= hour < 12:
        return 'Morning (5-12)'
    elif 12 <= hour < 17:
        return 'Afternoon (12-17)'
    elif 17 <= hour < 21:
        return 'Evening (17-21)'
    else:
        return 'Night (21-5)'

df['time_of_day'] = df['hour_of_day'].apply(time_bucket)

print("✅ Timestamp columns parsed successfully")
print(f"\n  Date range: {df['od_start_time'].min().date()} to {df['od_start_time'].max().date()}")
print(f"  Total days covered: {(df['od_start_time'].max() - df['od_start_time'].min()).days}")
print(f"\nTime of Day Distribution:")
print(df['time_of_day'].value_counts().to_string())

In [ ]:
# Delay ratio by time of day
time_delay = df.groupby('time_of_day')['delay_ratio'].agg(['mean', 'median', 'count'])
time_delay.columns = ['Mean Delay Ratio', 'Median Delay Ratio', 'Trip Count']
time_delay = time_delay.sort_values('Mean Delay Ratio', ascending=False)

print("Delay Ratio by Time of Day:\n")
print(time_delay.round(3).to_string())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean delay by time of day
colors_tod = ['#E05C5C' if v == time_delay['Mean Delay Ratio'].max() 
              else '#4A90D9' for v in time_delay['Mean Delay Ratio']]

bars = axes[0].bar(
    time_delay.index,
    time_delay['Mean Delay Ratio'],
    color=colors_tod,
    edgecolor='white',
    width=0.5
)
axes[0].axhline(1.0, color='green', linewidth=2, linestyle='--', label='Perfect (1.0)')
for bar, val in zip(bars, time_delay['Mean Delay Ratio']):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.005,
        f'{val:.3f}',
        ha='center', fontsize=10, fontweight='bold'
    )
axes[0].set_title('Mean Delay Ratio by Time of Day', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Time of Day', fontsize=11)
axes[0].set_ylabel('Mean Delay Ratio', fontsize=11)
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].tick_params(axis='x', rotation=15)

# Trip volume by time of day
axes[1].bar(
    time_delay.index,
    time_delay['Trip Count'],
    color='#7BC8A4',
    edgecolor='white',
    width=0.5
)
axes[1].set_title('Trip Volume by Time of Day', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Time of Day', fontsize=11)
axes[1].set_ylabel('Number of Trips', fontsize=11)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Time of Day — Delay & Volume Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/visualisations/time_of_day_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

## 🔗 Step 13 — Corridor Level Analysis

A corridor is a unique source → destination pair.  
We look at which corridors have the highest delay ratios —  
these will become the red edges in our graph visualization.

In [ ]:
# Corridor level delay analysis
corridor_stats = df.groupby([source_col, dest_col]).agg(
    trip_count   = ('delay_ratio', 'count'),
    mean_delay   = ('delay_ratio', 'mean'),
    median_delay = ('delay_ratio', 'median'),
    max_delay    = ('delay_ratio', 'max'),
    chronic_count= ('delay_ratio', lambda x: (x > 1.2).sum())
).reset_index()

corridor_stats['chronic_rate'] = (
    corridor_stats['chronic_count'] / corridor_stats['trip_count'] * 100
)

# Sort by mean delay
top_delayed = corridor_stats.sort_values('mean_delay', ascending=False).head(15)

print("Top 15 Most Delayed Corridors:\n")
print("-" * 80)
for _, row in top_delayed.iterrows():
    print(f"  {row[source_col][:20]:<20} → {row[dest_col][:20]:<20} | "
          f"Delay: {row['mean_delay']:.3f} | "
          f"Trips: {row['trip_count']:>4} | "
          f"Chronic: {row['chronic_rate']:.1f}%")
print("-" * 80)
print(f"\nTotal corridors analyzed: {len(corridor_stats):,}")

In [ ]:
# Top 15 delayed corridors visualization
fig, ax = plt.subplots(figsize=(12, 7))

corridor_labels = [
    f"{row[source_col][:15]}→{row[dest_col][:15]}" 
    for _, row in top_delayed.iterrows()
]

colors_corr = [
    '#E05C5C' if v > 1.5 else '#E8A838' if v > 1.2 else '#4A90D9' 
    for v in top_delayed['mean_delay']
]

bars = ax.barh(
    corridor_labels[::-1],
    top_delayed['mean_delay'].values[::-1],
    color=colors_corr[::-1],
    edgecolor='white',
    height=0.6
)

ax.axvline(1.0, color='green',  linewidth=2, linestyle='--', label='Perfect (1.0)')
ax.axvline(1.2, color='red',    linewidth=2, linestyle='--', label='Chronic threshold (1.2)')

ax.set_xlabel('Mean Delay Ratio', fontsize=12)
ax.set_title('Top 15 Most Delayed Corridors\n(Red = >1.5, Orange = 1.2-1.5, Blue = <1.2)',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/visualisations/top_delayed_corridors.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

## 🏆 Step 14 — Hub Level Overview

Before building the full graph, let's identify which hubs  
appear most frequently as source and destination.  
The most connected hubs will likely be our bottleneck candidates.

In [ ]:
# Hub frequency analysis
source_freq = df[source_col].value_counts().head(15)
dest_freq   = df[dest_col].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top source hubs
axes[0].barh(
    source_freq.index[::-1],
    source_freq.values[::-1],
    color='#4A90D9',
    edgecolor='white',
    height=0.6
)
axes[0].set_title('Top 15 Source Hubs by Trip Volume',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Trips', fontsize=11)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].tick_params(axis='y', labelsize=8)

# Top destination hubs
axes[1].barh(
    dest_freq.index[::-1],
    dest_freq.values[::-1],
    color='#E07B54',
    edgecolor='white',
    height=0.6
)
axes[1].set_title('Top 15 Destination Hubs by Trip Volume',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Trips', fontsize=11)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
axes[1].tick_params(axis='y', labelsize=8)

plt.suptitle('Hub Activity Overview', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/visualisations/hub_activity.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

## 📊 Step 15 — Segment vs Full Trip Delay Comparison

The dataset has both segment-level and full trip-level time columns.  
We compare `segment_actual_time` vs `actual_time` to understand  
how much of the delay happens within individual segments vs the full journey.

In [ ]:
# Segment vs full trip delay
df['segment_delay_ratio'] = df['segment_actual_time'] / df['segment_osrm_time']
df['segment_delay_ratio'] = df['segment_delay_ratio'].replace([np.inf, -np.inf], np.nan)

print("Full Trip Delay Ratio vs Segment Delay Ratio:\n")
print("-" * 45)
print(f"  Full trip mean delay   : {df['delay_ratio'].mean():.4f}")
print(f"  Segment mean delay     : {df['segment_delay_ratio'].mean():.4f}")
print(f"  Difference             : {df['delay_ratio'].mean() - df['segment_delay_ratio'].mean():.4f}")
print("-" * 45)

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(df['delay_ratio'].dropna(), bins=50, alpha=0.6,
        color='#4A90D9', label='Full Trip Delay Ratio', edgecolor='white')
ax.hist(df['segment_delay_ratio'].dropna(), bins=50, alpha=0.6,
        color='#E07B54', label='Segment Delay Ratio', edgecolor='white')

ax.axvline(1.0, color='green', linewidth=2, linestyle='--', label='Perfect (1.0)')
ax.set_xlabel('Delay Ratio', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Full Trip vs Segment Delay Distribution', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/visualisations/segment_vs_trip_delay.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Chart saved")

## 📋 Step 16 — Complete Findings Summary

Everything we've discovered in this notebook,  
documented clearly before moving to cleaning.

In [ ]:
print("=" * 65)
print("        COMPLETE EXPLORATION FINDINGS SUMMARY")
print("=" * 65)

print(f"""
DATASET STRUCTURE
  • Total segment records    : {len(df):,}
  • Unique trips             : {df['trip_uuid'].nunique():,}
  • Avg segments per trip    : {len(df)/df['trip_uuid'].nunique():.2f}
  • Columns                  : {df.shape[1]}
  • Date range               : {df['od_start_time'].min().date()} → {df['od_start_time'].max().date()}

NETWORK SCALE
  • Unique source hubs       : {df[source_col].nunique():,}
  • Unique destination hubs  : {df[dest_col].nunique():,}
  • Total unique hubs        : {pd.concat([df[source_col], df[dest_col]]).nunique():,}
  • Unique corridors         : {df.groupby([source_col, dest_col]).ngroups:,}
  • States covered           : {df['source_state'].nunique()}

ROUTE TYPE SPLIT
  • FTL                      : {(df[route_col]=='FTL').sum():,} ({(df[route_col]=='FTL').mean()*100:.1f}%)
  • Carting                  : {(df[route_col]=='Carting').sum():,} ({(df[route_col]=='Carting').mean()*100:.1f}%)

DELAY ANALYSIS (FULL TRIP)
  • Mean delay ratio         : {df['delay_ratio'].mean():.4f}
  • Median delay ratio       : {df['delay_ratio'].median():.4f}
  • % trips with ratio > 1.0 : {(df['delay_ratio'] > 1.0).mean()*100:.1f}%
  • Chronic delay rate (>1.2): {(df['delay_ratio'] > 1.2).mean()*100:.1f}%
  • OSRM underestimates      : {'YES ✅' if df['delay_ratio'].mean() > 1 else 'NO ❌'}

DELAY ANALYSIS (SEGMENT LEVEL)
  • Mean segment delay ratio : {df['segment_delay_ratio'].mean():.4f}

WORST TIME OF DAY FOR DELAYS
  • {time_delay['Mean Delay Ratio'].idxmax()} has highest mean delay ratio of {time_delay['Mean Delay Ratio'].max():.3f}

NEW COLUMNS CREATED
  • delay_ratio              : actual_time / osrm_time
  • segment_delay_ratio      : segment_actual_time / segment_osrm_time  
  • source_state             : extracted from source_name
  • dest_state               : extracted from destination_name
  • hour_of_day              : extracted from od_start_time
  • time_of_day              : Morning / Afternoon / Evening / Night
""")

print("=" * 65)
print("  VISUALIZATIONS SAVED TO: outputs/visualisations/")
print("  NEXT STEP → 02_data_cleaning.ipynb")
print("=" * 65)

---
## ✅ Data Exploration Complete

### What we confirmed:
1. Dataset has **segment-level** structure — multiple rows per trip
2. Network spans **pan-India** across multiple states
3. **OSRM consistently underestimates** actual delivery time — problem confirmed
4. Chronic delays affect a significant portion of trips
5. **FTL and Carting** show different delay profiles — critical for Phase 6
6. Certain **time windows** are significantly worse for delays
7. Top delayed corridors identified — these become priority targets

### New columns added this notebook:
- `delay_ratio`, `segment_delay_ratio`, `source_state`, `dest_state`, `hour_of_day`, `time_of_day`

### All charts saved to `outputs/visualisations/`

---
### ➡️ Next: `02_data_cleaning.ipynb`

In [ ]:
print("Delay Ratio Outlier Analysis:\n")
print(f"  Values > 10   : {(df['delay_ratio'] > 10).sum():,}")
print(f"  Values > 50   : {(df['delay_ratio'] > 50).sum():,}")
print(f"  Values > 100  : {(df['delay_ratio'] > 100).sum():,}")
print(f"  Values > 500  : {(df['delay_ratio'] > 500).sum():,}")
print(f"  Max value     : {df['delay_ratio'].max():.2f}")
print(f"\n  Segment ratio > 10  : {(df['segment_delay_ratio'] > 10).sum():,}")
print(f"  Segment ratio max   : {df['segment_delay_ratio'].max():.2f}")

"Segment-level delay ratios contain extreme outliers up to 574x, suggesting facility dwell time or data recording errors are inflating segment times. Full trip ratios are more contained (max 77x) indicating these extreme delays partially cancel out across multi-segment journeys."